In [ ]:
# Change to the san-v2 directory (adjust path as needed)
import os
os.chdir('/home/dgkagramanyan/san-v2')
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Check environment and GPU availability
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# Imports
import os
import re
import json
import tempfile
import torch
import legacy

import dnnlib
from training import training_loop
from metrics import metric_main
from torch_utils import training_stats
from torch_utils import custom_ops
from torch_utils import misc

# Import subprocess_fn from external module (required for multiprocessing in notebooks)
from notebook_utils import subprocess_fn

print("All imports successful!")

In [ ]:
#----------------------------------------------------------------------------
# Helper Functions
# Note: subprocess_fn is imported from notebook_utils.py (required for multiprocessing)
#----------------------------------------------------------------------------

def launch_training(c, desc, outdir, dry_run=False):
    """Launch the training process."""
    dnnlib.util.Logger(should_flush=True)

    # Pick output directory.
    prev_run_dirs = []
    if os.path.isdir(outdir):
        prev_run_dirs = [x for x in os.listdir(outdir) if os.path.isdir(os.path.join(outdir, x))]

    matching_dirs = [re.fullmatch(r'\d{5}' + f'-{desc}', x) for x in prev_run_dirs if re.fullmatch(r'\d{5}' + f'-{desc}', x) is not None]
    if c.restart_every > 0 and len(matching_dirs) > 0:  # expect unique desc, continue in this directory
        assert len(matching_dirs) == 1, f'Multiple directories found for resuming: {matching_dirs}'
        c.run_dir = os.path.join(outdir, matching_dirs[0].group())
    else:                     # fallback to standard
        prev_run_ids = [re.match(r'^\d+', x) for x in prev_run_dirs]
        prev_run_ids = [int(x.group()) for x in prev_run_ids if x is not None]
        cur_run_id = max(prev_run_ids, default=-1) + 1
        c.run_dir = os.path.join(outdir, f'{cur_run_id:05d}-{desc}')
        assert not os.path.exists(c.run_dir)

    # Print options.
    print()
    print('Training options:')
    print(json.dumps(c, indent=2))
    print()
    print(f'Output directory:    {c.run_dir}')
    print(f'Number of GPUs:      {c.num_gpus}')
    print(f'Batch size:          {c.batch_size} images')
    print(f'Training duration:   {c.total_kimg} kimg')
    print(f'Dataset path:        {c.training_set_kwargs.path}')
    print(f'Dataset size:        {c.training_set_kwargs.max_size} images')
    print(f'Dataset resolution:  {c.training_set_kwargs.resolution}')
    print(f'Dataset labels:      {c.training_set_kwargs.use_labels}')
    print(f'Dataset x-flips:     {c.training_set_kwargs.xflip}')
    print()

    # Dry run?
    if dry_run:
        print('Dry run; exiting.')
        return

    # Create output directory.
    print('Creating output directory...')
    os.makedirs(c.run_dir, exist_ok=c.restart_every > 0)
    with open(os.path.join(c.run_dir, 'training_options.json'), 'wt+') as f:
        json.dump(c, f, indent=2)

    # Launch processes.
    print('Launching processes...')
    try:
        torch.multiprocessing.set_start_method('spawn')
    except RuntimeError:
        pass  # Already set
    
    with tempfile.TemporaryDirectory() as temp_dir:
        if c.num_gpus == 1:
            subprocess_fn(rank=0, c=c, temp_dir=temp_dir)
        else:
            torch.multiprocessing.spawn(fn=subprocess_fn, args=(c, temp_dir), nprocs=c.num_gpus)

#----------------------------------------------------------------------------

def init_dataset_kwargs(data):
    """Initialize dataset configuration."""
    try:
        dataset_kwargs = dnnlib.EasyDict(class_name='training.dataset.ImageFolderDataset', path=data, use_labels=True, max_size=None, xflip=False)
        dataset_obj = dnnlib.util.construct_class_by_name(**dataset_kwargs)  # Subclass of training.dataset.Dataset.
        dataset_kwargs.resolution = dataset_obj.resolution  # Be explicit about resolution.
        dataset_kwargs.use_labels = dataset_obj.has_labels  # Be explicit about labels.
        dataset_kwargs.max_size = len(dataset_obj)  # Be explicit about dataset size.
        return dataset_kwargs, dataset_obj.name
    except IOError as err:
        raise Exception(f'Dataset error: {err}')

#----------------------------------------------------------------------------

def parse_comma_separated_list(s):
    """Parse comma-separated list of values."""
    if isinstance(s, list):
        return s
    if s is None or s.lower() == 'none' or s == '':
        return []
    return s.split(',')

print("Helper functions defined successfully!")

In [ ]:
#############################################################################
# TRAINING CONFIGURATION
# Modify the parameters below to configure your training run
#############################################################################

# Configuration parameters (equivalent to command line arguments in train.py)
opts = dnnlib.EasyDict({
    # Required parameters
    'outdir': './runs/wc-cv_h200',                              # Output directory for results
    'cfg': 'stylegan3-r',                                       # Base config: 'stylegan3-t', 'stylegan3-r', 'stylegan2', 'fastgan'
    'data': './datasets/imagenet_9to4_1024x1024_16x16.zip',     # Training data path (ZIP or DIR)
    'gpus': 4,                                                  # Number of GPUs to use
    'batch': 1280,                                              # Total batch size
    
    # Optional features
    'cond': True,                                               # Train conditional model
    'mirror': False,                                            # Enable dataset x-flips
    'resume': None,                                             # Resume from given network pickle (PATH or URL)
    'freezed': 0,                                               # Freeze first layers of D
    
    # Misc hyperparameters
    'batch_gpu': 320,                                           # Limit batch size per GPU (None = auto)
    'cbase': 32768,                                             # Capacity multiplier
    'cmax': 512,                                                # Max feature maps
    'glr': None,                                                # G learning rate (None = varies by config)
    'dlr': 0.002,                                               # D learning rate
    'map_depth': None,                                          # Mapping network depth (None = varies)
    
    # Misc settings
    'desc': None,                                               # String to include in result dir name
    'metrics': ['fid50k_full'],                                 # Quality metrics
    'kimg': 20000,                                              # Total training duration in kimg
    'tick': 4,                                                  # How often to print progress (kimg)
    'snap': 500,                                                # How often to save snapshots (ticks)
    'seed': 0,                                                  # Random seed
    'fp32': False,                                              # Disable mixed-precision
    'nobench': False,                                           # Disable cuDNN benchmarking
    'workers': 3,                                               # DataLoader worker processes
    'dry_run': False,                                           # Print training options and exit (set to True to preview)
    
    # StyleGAN-XL additions
    'restart_every': 999999999,                                 # Time interval in seconds to restart code
    'stem': False,                                              # Train the stem
    'syn_layers': 6,                                            # Number of layers in the stem
    'superres': False,                                          # Train superresolution stage
    'path_stem': None,                                          # Path to pretrained stem (required if superres=True)
    'head_layers': 7,                                           # Layers of added superresolution head
    'cls_weight': 0.0,                                          # Class guidance weight
    'up_factor': 2,                                             # Up sampling factor of superres head
})

print("Configuration loaded:")
print(f"  Output dir: {opts.outdir}")
print(f"  Config: {opts.cfg}")
print(f"  Dataset: {opts.data}")
print(f"  GPUs: {opts.gpus}")
print(f"  Batch size: {opts.batch}")
print(f"  Conditional: {opts.cond}")
print(f"  Training kimg: {opts.kimg}")


In [ ]:
#############################################################################
# BUILD TRAINING CONFIG
# This cell processes opts and builds the full training configuration
#############################################################################

# Initialize config dict
c = dnnlib.EasyDict()
c.G_kwargs = dnnlib.EasyDict(class_name=None, z_dim=512, w_dim=512, mapping_kwargs=dnnlib.EasyDict())
c.G_opt_kwargs = dnnlib.EasyDict(class_name='torch.optim.Adam', betas=[0, 0.99], eps=1e-8)
c.D_opt_kwargs = dnnlib.EasyDict(class_name='torch.optim.Adam', betas=[0, 0.99], eps=1e-8)
c.data_loader_kwargs = dnnlib.EasyDict(pin_memory=True, prefetch_factor=2)

# Training set
c.training_set_kwargs, dataset_name = init_dataset_kwargs(data=opts.data)
if opts.cond and not c.training_set_kwargs.use_labels:
    raise Exception('cond=True requires labels specified in dataset.json')
c.training_set_kwargs.use_labels = opts.cond
c.training_set_kwargs.xflip = opts.mirror

# Hyperparameters & settings
c.num_gpus = opts.gpus
c.batch_size = opts.batch
c.batch_gpu = opts.batch_gpu or opts.batch // opts.gpus
c.G_kwargs.channel_base = opts.cbase
c.G_kwargs.channel_max = opts.cmax
c.G_opt_kwargs.lr = (0.002 if opts.cfg == 'stylegan2' else 0.0025) if opts.glr is None else opts.glr
c.D_opt_kwargs.lr = opts.dlr
c.metrics = opts.metrics
c.total_kimg = opts.kimg
c.kimg_per_tick = opts.tick
c.image_snapshot_ticks = c.network_snapshot_ticks = opts.snap
c.random_seed = c.training_set_kwargs.random_seed = opts.seed
c.data_loader_kwargs.num_workers = opts.workers

# Sanity checks
if c.batch_size % c.num_gpus != 0:
    raise Exception('batch must be a multiple of gpus')
if c.batch_size % (c.num_gpus * c.batch_gpu) != 0:
    raise Exception('batch must be a multiple of gpus times batch_gpu')
if any(not metric_main.is_valid_metric(metric) for metric in c.metrics):
    raise Exception('\n'.join(['metrics can only contain the following values:'] + metric_main.list_valid_metrics()))

# Base configuration
c.ema_kimg = c.batch_size * 10 / 32

if opts.cfg == 'stylegan2':
    c.G_kwargs.class_name = 'training.networks_stylegan2.Generator'
    c.G_reg_interval = 4  # Enable lazy regularization for G
    c.G_kwargs.fused_modconv_default = 'inference_only'

elif opts.cfg == 'fastgan':
    c.G_kwargs = dnnlib.EasyDict(
        class_name='training.networks_fastgan.Generator',
        cond=opts.cond, 
        mapping_kwargs=dnnlib.EasyDict(),
        synthesis_kwargs=dnnlib.EasyDict()
    )
    c.G_kwargs.synthesis_kwargs.lite = True
    c.G_opt_kwargs.lr = c.D_opt_kwargs.lr = 0.0002
    c.G_opt_kwargs.lr = 0.002

else:  # stylegan3-t or stylegan3-r
    c.G_kwargs.class_name = 'training.networks_stylegan3_resetting.Generator'
    c.G_kwargs.magnitude_ema_beta = 0.5 ** (c.batch_size / (20 * 1e3))
    c.G_kwargs.channel_base *= 2  # increase for StyleGAN-XL
    c.G_kwargs.channel_max *= 2   # increase for StyleGAN-XL
    c.G_kwargs.conv_kernel = 1 if opts.cfg == 'stylegan3-r' else 3
    c.G_kwargs.use_radial_filters = True if opts.cfg == 'stylegan3-r' else False

    if opts.cfg == 'stylegan3-r':
        c.G_kwargs.channel_base *= 2
        c.G_kwargs.channel_max *= 2

# Resume
if opts.resume is not None:
    c.resume_pkl = opts.resume
    c.ada_kimg = 100  # Make ADA react faster at the beginning
    c.ema_rampup = None  # Disable EMA rampup

# Restart
c.restart_every = opts.restart_every

# Performance-related toggles
if opts.fp32:
    c.G_kwargs.num_fp16_res = 0
    c.G_kwargs.conv_clamp = None
if opts.nobench:
    c.cudnn_benchmark = False

# Description string
desc = f'{opts.cfg:s}-{dataset_name:s}-gpus{c.num_gpus:d}-batch{c.batch_size:d}'
if opts.desc is not None:
    desc += f'-{opts.desc}'

##################################
########## StyleGAN-XL ###########
##################################

# Generator settings
c.G_kwargs.w_dim = 512
c.G_kwargs.z_dim = 64
c.G_kwargs.mapping_kwargs.rand_embedding = False
c.G_kwargs.num_layers = opts.syn_layers
c.G_kwargs.mapping_kwargs.num_layers = 2

# Discriminator settings
c.D_kwargs = dnnlib.EasyDict(
    class_name='pg_modules.discriminator.ProjectedDiscriminator',
    backbones=['deit_base_distilled_patch16_224', 'tf_efficientnet_lite0'],
    diffaug=True,
    interp224=(c.training_set_kwargs.resolution < 224),
    backbone_kwargs=dnnlib.EasyDict(),
)
c.D_kwargs.backbone_kwargs.cout = 64
c.D_kwargs.backbone_kwargs.expand = True
c.D_kwargs.backbone_kwargs.proj_type = 2 if c.training_set_kwargs.resolution <= 16 else 2
c.D_kwargs.backbone_kwargs.num_discs = 4
c.D_kwargs.backbone_kwargs.cond = opts.cond

# Loss settings
c.loss_kwargs = dnnlib.EasyDict(class_name='training.loss.ProjectedGANLoss')
c.loss_kwargs.blur_init_sigma = 2
c.loss_kwargs.blur_fade_kimg = 300
c.loss_kwargs.pl_weight = 2.0
c.loss_kwargs.pl_no_weight_grad = True
c.loss_kwargs.style_mixing_prob = 0.0
c.loss_kwargs.cls_weight = 0.0
c.loss_kwargs.cls_model = 'deit_small_distilled_patch16_224'
c.loss_kwargs.train_head_only = False

# Superresolution settings (if enabled)
if opts.superres:
    assert opts.path_stem is not None, "When training superres head, provide path to stem"
    
    c.G_kwargs = dnnlib.EasyDict(
        class_name='training.networks_stylegan3_resetting.SuperresGenerator',
        path_stem=opts.path_stem,
        head_layers=opts.head_layers,
        up_factor=opts.up_factor,
        conv_kernel=1 if opts.cfg == 'stylegan3-r' else 3,
        use_radial_filters=True if opts.cfg == 'stylegan3-r' else False,
    )
    
    c.loss_kwargs.pl_weight = 0.0
    c.loss_kwargs.cls_weight = opts.cls_weight if opts.cond else 0
    c.loss_kwargs.train_head_only = True

print("Training configuration built successfully!")
print(f"  Dataset: {dataset_name}")
print(f"  Resolution: {c.training_set_kwargs.resolution}")
print(f"  Dataset size: {c.training_set_kwargs.max_size} images")
print(f"  Description: {desc}")


In [ ]:
#############################################################################
# LAUNCH TRAINING
# Run this cell to start training
# Set opts.dry_run = True in the config cell to preview settings without training
#############################################################################

# Launch the training
launch_training(c=c, desc=desc, outdir=opts.outdir, dry_run=opts.dry_run)

# Check for restart (useful for long training runs)
try:
    last_snapshot = misc.get_ckpt_path(c.run_dir)
    if os.path.isfile(last_snapshot):
        with dnnlib.util.open_url(last_snapshot) as f:
            cur_nimg = legacy.load_network_pkl(f)['progress']['cur_nimg'].item()
        if (cur_nimg // 1000) < c.total_kimg:
            print(f'\nTraining progress: {cur_nimg // 1000} / {c.total_kimg} kimg')
            print('Training incomplete - consider resuming by re-running this cell')
        else:
            print(f'\nTraining complete! {cur_nimg // 1000} kimg reached.')
except Exception as e:
    print(f"Could not check training progress: {e}")


In [ ]:
# Pre-compile CUDA extensions (run this once before multi-GPU training)
import os
os.chdir('/home/dgkagramanyan/san-v2')
print(f"Working directory: {os.getcwd()}")

import torch
from torch_utils import custom_ops

# Set verbosity to see compilation progress
custom_ops.verbosity = 'full'

# Force compilation by importing the ops
from torch_utils.ops import upfirdn2d
from torch_utils.ops import bias_act
from torch_utils.ops import filtered_lrelu

# Trigger compilation
print("Compiling upfirdn2d...")
upfirdn2d._init()

print("Compiling bias_act...")
bias_act._init()

print("Compiling filtered_lrelu...")
filtered_lrelu._init()

print("Pre-compilation complete!")

: 

In [ ]:
!pip uninstall ninja -y

In [ ]:
!conda install -c conda-forge ninja


In [ ]:
!conda install anaconda::ninja -y

In [ ]:
!conda uninstall ninja -y

In [ ]:
!conda list torch